In [0]:
# Journal Risk: gold table + UC Function tools (SINGLE CELL version)
# Clear the existing cell, paste ALL of this into one cell, Run.
# pip install is handled at top; no separate cells needed.
 
%pip install scikit-learn pandas numpy --quiet
 
# ---- after pip, the rest runs in the same flow ----
from decimal import Decimal
import pandas as pd, numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pyspark.sql import functions as F
 
TABLE_PATH = "bdc_share_journal_entry.entryviewjournalentry.operationalacctgdocitem"
cols = ["CompanyCode","AccountingDocument","FiscalYear","GLAccount",
        "PostingDate","DebitCreditCode","IsAutomaticallyCreated",
        "AmountInCompanyCodeCurrency"]
 
df = spark.table(TABLE_PATH).select(cols).limit(50000).toPandas()
for c in df.columns:
    df[c] = df[c].apply(lambda x: float(x) if isinstance(x, Decimal) else x)
 
df['PostingDate'] = pd.to_datetime(df['PostingDate'], errors='coerce')
df['is_weekend']  = df['PostingDate'].dt.dayofweek.isin([5,6]).astype(int)
df['abs_amount']  = df['AmountInCompanyCodeCurrency'].abs()
df['log_abs']     = np.log1p(df['abs_amount'])
df['is_negative'] = (df['AmountInCompanyCodeCurrency'] < 0).astype(int)
df['is_manual']   = (~df['IsAutomaticallyCreated'].astype(bool)).astype(int)
df['is_debit']    = (df['DebitCreditCode'] == 'S').astype(int)
for c in ['CompanyCode','GLAccount']:
    df[c+'_enc'] = LabelEncoder().fit_transform(df[c].astype(str))
 
feats = ['abs_amount','log_abs','is_negative','is_weekend','is_manual',
         'is_debit','CompanyCode_enc','GLAccount_enc']
X = df[feats].replace([np.inf,-np.inf], np.nan)
X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42, n_jobs=-1)
df['is_anomaly'] = (iso.fit_predict(Xs) == -1).astype(int)
print("Anomaly rate:", round(100*df['is_anomaly'].mean(),2), "%")
 
# ---- gold table ----
sdf = spark.createDataFrame(df[[
    'CompanyCode','AccountingDocument','FiscalYear','GLAccount',
    'AmountInCompanyCodeCurrency','is_weekend','is_manual','is_negative','is_anomaly'
]]).toDF(
    'company_code','accounting_document','fiscal_year','gl_account',
    'amount_company_currency','is_weekend','is_manual','is_negative','is_anomaly'
)
counts = sdf.groupBy('company_code').count().withColumnRenamed('count','entry_count')
gold = (sdf.join(counts,'company_code')
    .withColumn('amount_company_currency', F.col('amount_company_currency').cast('decimal(38,4)'))
    .withColumn('data_confidence',
        F.when(F.col('entry_count') < 100, F.lit('low')).otherwise(F.lit('normal'))))
gold.write.mode('overwrite').option('overwriteSchema','true').saveAsTable('workspace.default.gold_journal_risk')
print("gold_journal_risk created:", gold.count(), "rows")
 
# ---- tool 1 ----
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_journal_risk(
    p_company_code STRING COMMENT 'SAP company code, e.g. 1010 or 1710'
)
RETURNS TABLE (
    company_code STRING, total_entries BIGINT, anomalies_flagged BIGINT,
    anomaly_rate_pct DECIMAL(10,2), weekend_anomaly_pct DECIMAL(10,2),
    manual_anomaly_pct DECIMAL(10,2), data_confidence STRING
)
COMMENT 'Returns journal entry control-risk summary for a SAP company code. Anomalies are statistical flags for review, not confirmed fraud. Always surface data_confidence.'
RETURN
    SELECT company_code, COUNT(*) AS total_entries, SUM(is_anomaly) AS anomalies_flagged,
        ROUND(100.0*SUM(is_anomaly)/COUNT(*),2) AS anomaly_rate_pct,
        ROUND(100.0*SUM(CASE WHEN is_anomaly=1 AND is_weekend=1 THEN 1 ELSE 0 END)/NULLIF(SUM(is_anomaly),0),2) AS weekend_anomaly_pct,
        ROUND(100.0*SUM(CASE WHEN is_anomaly=1 AND is_manual=1 THEN 1 ELSE 0 END)/NULLIF(SUM(is_anomaly),0),2) AS manual_anomaly_pct,
        MAX(data_confidence) AS data_confidence
    FROM workspace.default.gold_journal_risk
    WHERE company_code = p_company_code GROUP BY company_code
""")
 
# ---- tool 2 ----
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rank_journal_risk(
    p_min_entries INT COMMENT 'Minimum entries for a company code to be ranked'
)
RETURNS TABLE (company_code STRING, total_entries BIGINT, anomaly_rate_pct DECIMAL(10,2))
COMMENT 'Ranks SAP company codes by journal anomaly rate, excluding codes with fewer than p_min_entries.'
RETURN
    SELECT company_code, COUNT(*) AS total_entries,
        ROUND(100.0*SUM(is_anomaly)/COUNT(*),2) AS anomaly_rate_pct
    FROM workspace.default.gold_journal_risk
    GROUP BY company_code HAVING COUNT(*) >= p_min_entries
    ORDER BY anomaly_rate_pct DESC
""")
 
# ---- tests ----
print("Test get_journal_risk('1710'):")
spark.sql("SELECT * FROM workspace.default.get_journal_risk('1710')").show()
print("Test rank_journal_risk(100):")
spark.sql("SELECT * FROM workspace.default.rank_journal_risk(100)").show()
 